In [1]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 52.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.3/934.3 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 107.8 MB/s eta 0:00:0000:0100:01


In [2]:
import os, re, time, random
from pathlib import Path
from collections import defaultdict, OrderedDict
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pennylane as qml
import torchvision.models as models
from torchvision.models import EfficientNet_B0_Weights
from torchvision import transforms
import cv2
import numpy as np
import time
import random
from copy import deepcopy

In [3]:
DATASET_ROOT = "/kaggle/input/breakhis/BreaKHis_v1/BreaKHis_v1/histology_slides/breast"  
NUM_WORKERS = 4
NUM_EPOCHS = 25
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_PATH = "best_hqcnn_quantum.pth"
SPLIT_RATIO = 0.8   
RANDOM_SEED = 42

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}
PAT_200 = re.compile(r'200', re.IGNORECASE)
PAT_400 = re.compile(r'400', re.IGNORECASE)

In [4]:
class CLAHETransform:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8,8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    def __call__(self, img):
        img = np.array(img)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(img)
        l2 = self.clahe.apply(l)
        img = cv2.merge([l2, a, b])
        img = cv2.cvtColor(img, cv2.COLOR_LAB2RGB)
        return Image.fromarray(img)

In [5]:
train_tf = transforms.Compose([
    CLAHETransform(),                     
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_tf = transforms.Compose([
    CLAHETransform(),             
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])


In [6]:
def list_all_images(root: Path):
    imgs = []
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            imgs.append(p)
    return imgs

def infer_main_and_subclass(path: Path, main_candidates=('benign','malignant')):
    parts = [p.lower() for p in path.parts]
    main_idx = None
    main = None
    for candidate in main_candidates:
        if candidate in parts:
            main_idx = parts.index(candidate)
            main = parts[main_idx]
            break
    if main is None:
        return None, None

    return main, None  

def build_subclass_candidates(root: Path, main_class: str):
    base = root / main_class
    candidates = set()
    if not base.exists():
        return candidates
    try:
        for child in base.iterdir():
            if child.is_dir():
                candidates.add(child.name.lower())
                for grand in child.iterdir():
                    if grand.is_dir():
                        candidates.add(grand.name.lower())
                        for g2 in grand.iterdir():
                            if g2.is_dir():
                                candidates.add(g2.name.lower())
    except Exception:
        pass
    return candidates

def extract_subclass_from_path(path: Path, main_class: str, subclass_candidates: set):
    parts = [p for p in path.parts]
    parts_low = [p.lower() for p in parts]
    try:
        main_idx = parts_low.index(main_class)
    except ValueError:
        return None

    for p in parts_low[main_idx+1:]:
        if p in subclass_candidates:
            return p

    mag_idx = None
    for i,p in enumerate(parts_low):
        if PAT_200.search(p) or PAT_400.search(p):
            mag_idx = i
            break

    if mag_idx:
        guess_idx = mag_idx - 3
        if guess_idx > main_idx and guess_idx < len(parts):
            return parts_low[guess_idx]
        guess_idx = mag_idx - 2
        if guess_idx > main_idx and guess_idx < len(parts):
            return parts_low[guess_idx]

    if main_idx + 1 < len(parts):
        return parts_low[main_idx+1]
    return None

In [7]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}

PAT_40  = re.compile(r'40x', re.IGNORECASE)
PAT_100 = re.compile(r'100x', re.IGNORECASE)
PAT_200 = re.compile(r'200x', re.IGNORECASE)
PAT_400 = re.compile(r'400x', re.IGNORECASE)

MAG_PATTERNS = {
    "40X": PAT_40,
    "100X": PAT_100,
    "200X": PAT_200,
    "400X": PAT_400
}

MAG_KEYS = list(MAG_PATTERNS.keys())

In [8]:
root = Path(DATASET_ROOT)
if not root.exists():
    raise FileNotFoundError(f"DATASET_ROOT not found: {root}")

sub_candidates = {
    'benign': build_subclass_candidates(root, 'benign'),
    'malignant': build_subclass_candidates(root, 'malignant')
}

all_images = list_all_images(root)
print(f"Total images discovered under {root}: {len(all_images)} (all magnifications)")

def make_bucket():
    return {m: [] for m in MAG_KEYS}

counters = {
    'benign': defaultdict(make_bucket),
    'malignant': defaultdict(make_bucket),
    'unknown': defaultdict(make_bucket)
}

for p in all_images:
    plow = str(p).lower()

    magnification = None
    for mkey, pattern in MAG_PATTERNS.items():
        if pattern.search(plow):
            magnification = mkey
            break

    if magnification is None:
        continue

    parts_low = [pp.lower() for pp in p.parts]
    if 'benign' in parts_low:
        main = 'benign'
    elif 'malignant' in parts_low:
        main = 'malignant'
    else:
        main = 'unknown'

    subclass = extract_subclass_from_path(p, main, sub_candidates.get(main, set()))
    if subclass is None:
        subclass = 'unknown_subclass'

    counters[main][subclass][magnification].append(str(p))

print("\n=== Counts per main class -> subclass -> magnification ===")
for main in ['benign','malignant','unknown']:
    print(f"\nMAIN CLASS: {main.upper()}")
    scounts = counters[main]
    if not scounts:
        print("  (no subclasses found)")
        continue

    for subclass in sorted(scounts.keys()):
        bucket = scounts[subclass]
        c40  = len(bucket["40X"])
        c100 = len(bucket["100X"])
        c200 = len(bucket["200X"])
        c400 = len(bucket["400X"])
        total = c40 + c100 + c200 + c400

        print(f"  Subclass: {subclass:30s}  total={total:4d}  "
              f"40X={c40:4d}  100X={c100:4d}  200X={c200:4d}  400X={c400:4d}")

Total images discovered under /kaggle/input/breakhis/BreaKHis_v1/BreaKHis_v1/histology_slides/breast: 7909 (all magnifications)

=== Counts per main class -> subclass -> magnification ===

MAIN CLASS: BENIGN
  Subclass: sob                             total=2480  40X= 625  100X= 644  200X= 623  400X= 588

MAIN CLASS: MALIGNANT
  Subclass: sob                             total=5429  40X=1370  100X=1437  200X=1390  400X=1232

MAIN CLASS: UNKNOWN
  (no subclasses found)


In [9]:
filepaths = []
labels = []
label_names = []
label_map = {}

selected_subclasses = []
for main in ('benign','malignant'):
    for subclass, buckets in counters[main].items():
        total_imgs = sum(len(buckets[m]) for m in MAG_KEYS)
        if total_imgs > 0:
            selected_subclasses.append((main, subclass))

if not selected_subclasses:
    raise RuntimeError("No magnification images found.")

for idx, (main, subclass) in enumerate(sorted(selected_subclasses)):
    label_map[(main, subclass)] = idx
    label_names.append(f"{main}/{subclass}")

for (main, subclass), idx in label_map.items():
    bucket = counters[main][subclass]
    for m in MAG_KEYS:
        for fp in bucket[m]:
            filepaths.append(fp)
            labels.append(idx)

print(f"\nTotal images selected for dataset (all magnifications): {len(filepaths)}")
print(f"Number of subclass labels: {len(label_names)}")
print("Label mapping (index -> main/subclass):")
for (main, subclass), idx in label_map.items():
    print(f"  {idx}: {main}/{subclass}")



Total images selected for dataset (all magnifications): 7909
Number of subclass labels: 2
Label mapping (index -> main/subclass):
  0: benign/sob
  1: malignant/sob


In [10]:
random.seed(RANDOM_SEED)
train_files, train_labels = [], []
val_files, val_labels = [], []

by_label = defaultdict(list)
for fp, lbl in zip(filepaths, labels):
    by_label[lbl].append(fp)

for lbl, fps in by_label.items():
    random.shuffle(fps)
    split = int(len(fps) * SPLIT_RATIO)
    if split == 0 and len(fps) > 1:
        split = 1
    train_part = fps[:split]
    val_part   = fps[split:]

    if len(val_part) == 0 and len(train_part) > 1:
        val_part = train_part[-1:]
        train_part = train_part[:-1]

    train_files += train_part
    train_labels += [lbl] * len(train_part)
    val_files += val_part
    val_labels += [lbl] * len(val_part)

print(f"\nAfter stratified split -> train: {len(train_files)}  val: {len(val_files)}")


After stratified split -> train: 6327  val: 1582


In [11]:
class FileDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

train_dataset = FileDataset(train_files, train_labels, transform=train_tf)
val_dataset   = FileDataset(val_files, val_labels, transform=val_tf)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

In [13]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_QUBITS = 6
NUM_LAYERS = 1
BASE_DEPOL = 0.02    
LR = 1e-4
WD = 1e-5
SAVE_PATH = "efficientnet6q_zne.pth"
ZNE_SCALES = (1.0, 3.0, 5.0)
ZNE_ORDER = 2

print("Device:", DEVICE)
import pennylane as qml
print("PennyLane version:", qml.__version__)

Device: cuda
PennyLane version: 0.43.2


In [14]:
from copy import deepcopy
import numpy as np
import torch
import torch.nn as nn
import pennylane as qml

class QuantumLayerZNE(nn.Module):
    def __init__(
        self,
        n_qubits=6,
        n_layers=1,
        base_depolarizing_prob=0.02,
        shots=1000
    ):
        super().__init__()

        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.base_depol = float(base_depolarizing_prob)
        self.shots = shots

        self.use_shots = False   
        self.weight_shapes = {"weights": (n_layers, n_qubits, 3)}

        self.device_name = "lightning.qubit" 
        self._current_noise_prob = -1.0

        self._make_qnode(noise_prob=0.0)
        self.qlayer = qml.qnn.TorchLayer(self.qnode, self.weight_shapes)

    # noise 
    def _apply_noise(self, p, wire):
        if p > 0:
            qml.DepolarizingChannel(p, wires=wire)

    # qnode 
    def _make_qnode(self, noise_prob):
        dev = qml.device(
            self.device_name,
            wires=self.n_qubits,
            shots=self.shots if self.use_shots else None
        )

        @qml.qnode(dev, interface="torch")
        def circuit(inputs, weights):
            qml.AngleEmbedding(
                torch.tanh(inputs) * np.pi,
                wires=range(self.n_qubits),
                rotation="Y"
            )

            for layer in range(self.n_layers):
                for i in range(self.n_qubits):
                    for j in range(i + 1, self.n_qubits):
                        qml.CNOT(wires=[i, j])
                        if self.use_shots:
                            self._apply_noise(noise_prob, j)

                for i in range(self.n_qubits):
                    qml.RX(weights[layer, i, 0], wires=i)
                    qml.RY(weights[layer, i, 1], wires=i)
                    qml.RZ(weights[layer, i, 2], wires=i)
                    if self.use_shots:
                        self._apply_noise(noise_prob, i)

            return [qml.expval(qml.PauliZ(i)) for i in range(self.n_qubits)]

        self.qnode = circuit

    def _ensure_layer(self, noise_prob):
        if abs(self._current_noise_prob - noise_prob) < 1e-12:
            return

        prev_state = deepcopy(self.qlayer.state_dict())

        self._make_qnode(noise_prob)
        self.qlayer = qml.qnn.TorchLayer(self.qnode, self.weight_shapes)
        self.qlayer.load_state_dict(prev_state)

        self._current_noise_prob = noise_prob

    # forwards
    def forward_single_scale(self, x, noise_scale=1.0):
        noise_prob = self.base_depol * noise_scale
        self._ensure_layer(noise_prob)
        
        if not self.use_shots:
            return self.qlayer(x)

        outs = []
        for i in range(x.shape[0]):
            outs.append(self.qlayer(x[i]))
        return torch.stack(outs, dim=0)

    def forward_zne(self, x, scales=(1.0, 3.0, 5.0), order=2):
        outs = [self.forward_single_scale(x, s) for s in scales]
        return self._richardson(outs, scales, order)

    @staticmethod
    def _richardson(y_list, scales, order):
        Y = np.stack([y.detach().cpu().numpy() for y in y_list])
        S = np.array(scales)
        deg = min(len(scales) - 1, order)
        V = np.vander(S, deg + 1)
        coeffs = np.linalg.pinv(V).dot(Y.reshape(len(scales), -1))
        y0 = coeffs[-1].reshape(Y.shape[1], Y.shape[2])
        return torch.tensor(y0, device=y_list[0].device, dtype=y_list[0].dtype)


In [15]:
class HQCNN_EfficientNet_Quantum_ZNE(nn.Module):
    def __init__(self, num_classes, n_qubits=6, n_layers=1, embed_dim=32,freeze_backbone=False, base_depol=0.02):   

        self.use_quantum = True
        self.n_qubits = n_qubits
        self.n_layers = n_layers

        # Backbone
        self.backbone = models.efficientnet_b0(
            weights=EfficientNet_B0_Weights.IMAGENET1K_V1
        )
        feat_dim = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        # Projection → Quantum
        self.proj = nn.Linear(feat_dim, n_qubits)
        self.quantum_layer = QuantumLayerZNE(
            n_qubits=n_qubits,
            n_layers=n_layers,
            base_depolarizing_prob=base_depol
        )

        # Post-quantum embedding + attention
        self.quantum_post = nn.Linear(n_qubits, embed_dim)
        self.att_fc1 = nn.Linear(embed_dim, max(1, embed_dim // 4))
        self.att_fc2 = nn.Linear(max(1, embed_dim // 4), embed_dim)

        # Classification head
        self.fc_out = nn.Linear(embed_dim, num_classes)

    def forward(self, x, zne=False, zne_scales=(1.0, 3.0, 5.0)):
        feat = self.backbone(x)
        proj = self.proj(feat)
    
        if zne:
            q_out = self.quantum_layer.forward_zne(
                proj, scales=zne_scales, order=ZNE_ORDER
            )
        else:
            q_out = self.quantum_layer.forward_single_scale(
                proj, noise_scale=1.0
            )
    
        q_out = q_out.to(self.fc_out.weight.device)
    
        q_feat = self.quantum_post(q_out)
        s = F.relu(self.att_fc1(q_feat))
        att = torch.sigmoid(self.att_fc2(s))
        attended = q_feat * att
    
        return self.fc_out(attended)



In [16]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    running_corrects = 0
    total = 0
    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        outputs = model(inputs, zne=False)   # training without ZNE for robustness and correctness
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        _, preds = torch.max(outputs, 1)
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        total += labels.size(0)
    return (running_loss/total) if total else 0.0, (running_corrects.double().item()/total) if total else 0.0

def evaluate(model, loader, device, zne=False, zne_scales=(1.0,3.0,5.0), max_batches=None):
    model.eval()
    running_corrects = 0
    total = 0
    t0 = time.time()
    with torch.no_grad():
        for i, (inputs, labels) in enumerate(loader):
            if max_batches and i >= max_batches:
                break
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(inputs, zne=zne, zne_scales=zne_scales)
            _, preds = torch.max(outputs, 1)
            running_corrects += torch.sum(preds == labels.data)
            total += labels.size(0)
    elapsed = time.time() - t0
    acc = (running_corrects.double().item()/total) if total else 0.0
    return acc, elapsed

In [17]:
try:
    train_loader  
    val_loader    
except NameError:
    raise RuntimeError("Please ensure train_loader and val_loader are defined in the notebook before running this cell.")

def infer_num_classes(loader):
    dataset = getattr(loader, "dataset", None)
    if dataset is None:
        return 2
    if hasattr(dataset, "labels"):
        return len(set(dataset.labels))
    sample_lbls = []
    for _, lbls in loader:
        sample_lbls.extend(lbls.tolist())
        if len(sample_lbls) >= 200:
            break
    return len(set(sample_lbls)) if sample_lbls else 2

NUM_CLASSES = infer_num_classes(train_loader)
print("Inferred num_classes:", NUM_CLASSES)

Inferred num_classes: 2


In [19]:
# Instantiating model, optimizer, scheduler 
model = HQCNN_EfficientNet_Quantum_ZNE(
    num_classes=NUM_CLASSES,
    n_qubits=NUM_QUBITS,
    n_layers=NUM_LAYERS,
    embed_dim=32,
    freeze_backbone=False,
    base_depol=BASE_DEPOL
).to(DEVICE)


# Parameter accounting
total_params = 0
trainable_params = 0
frozen_params = 0

backbone_params = 0
head_params = 0
quantum_params = 0

for name, param in model.named_parameters():
    n = param.numel()
    total_params += n

    if param.requires_grad:
        trainable_params += n
    else:
        frozen_params += n

    lname = name.lower()

    if "backbone" in lname or "efficientnet" in lname:
        backbone_params += n
    elif any(k in lname for k in ["quantum", "vqc", "q_layer", "qcnn"]):
        quantum_params += n
    else:
        head_params += n     

print("Parameter Summary")
print("=" * 55)
print(f"Total parameters        : {total_params:,}")
print(f"Trainable parameters    : {trainable_params:,}")
print(f"Frozen parameters       : {frozen_params:,}")
print("-" * 55)
print(f"Backbone parameters     : {backbone_params:,}")
print(f"Classical head params   : {head_params:,}")
print(f"Quantum parameters      : {quantum_params:,}")
print("-" * 55)
print("ZNE parameters          : 0 (inference-time only)")
print("=" * 55)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

Parameter Summary
Total parameters        : 4,016,094
Trainable parameters    : 4,016,094
Frozen parameters       : 0
-------------------------------------------------------
Backbone parameters     : 4,007,548
Classical head params   : 8,304
Quantum parameters      : 242
-------------------------------------------------------
ZNE parameters          : 0 (inference-time only)


In [23]:
best_val_acc = 0.0
print("\n=== QUICK TRAIN START ===")
for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)

    val_acc_subset, _ = evaluate(model, val_loader, DEVICE, zne=False, max_batches=50)
 
    try:
        scheduler.step(1.0 - val_acc_subset)
    except Exception:
        pass
    if val_acc_subset > best_val_acc:
        best_val_acc = val_acc_subset
        torch.save(model.state_dict(), SAVE_PATH)
    elapsed = time.time() - t0
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Train Loss: {train_loss:.4f} Train Acc: {train_acc:.4f} - Val Acc(subset,no ZNE): {val_acc_subset:.4f} - Time: {elapsed:.1f}s")
print("=== QUICK TRAIN FINISHED ===")
print("Best val acc (subset checks):", best_val_acc)
print("Saved best model to:", SAVE_PATH)

model.quantum_layer.use_shots = True
model.quantum_layer.device_name = "default.mixed"
model.quantum_layer._current_noise_prob = -1.0
model.quantum_layer._make_qnode(noise_prob=BASE_DEPOL)
model.quantum_layer.qlayer = qml.qnn.TorchLayer(
    model.quantum_layer.qnode,
    model.quantum_layer.weight_shapes
)



=== QUICK TRAIN START ===
Epoch 1/25 - Train Loss: 0.5321 Train Acc: 0.8453 - Val Acc(subset,no ZNE): 0.9368 - Time: 304.4s
Epoch 2/25 - Train Loss: 0.3465 Train Acc: 0.9339 - Val Acc(subset,no ZNE): 0.9703 - Time: 292.4s
Epoch 3/25 - Train Loss: 0.2205 Train Acc: 0.9573 - Val Acc(subset,no ZNE): 0.9785 - Time: 291.7s
Epoch 4/25 - Train Loss: 0.1360 Train Acc: 0.9757 - Val Acc(subset,no ZNE): 0.9741 - Time: 292.1s
Epoch 5/25 - Train Loss: 0.1050 Train Acc: 0.9736 - Val Acc(subset,no ZNE): 0.9551 - Time: 291.7s
Epoch 6/25 - Train Loss: 0.0707 Train Acc: 0.9847 - Val Acc(subset,no ZNE): 0.9848 - Time: 291.3s
Epoch 7/25 - Train Loss: 0.0540 Train Acc: 0.9867 - Val Acc(subset,no ZNE): 0.9848 - Time: 290.7s
Epoch 8/25 - Train Loss: 0.0395 Train Acc: 0.9899 - Val Acc(subset,no ZNE): 0.9810 - Time: 291.0s
Epoch 9/25 - Train Loss: 0.0390 Train Acc: 0.9891 - Val Acc(subset,no ZNE): 0.9874 - Time: 291.1s
Epoch 10/25 - Train Loss: 0.0306 Train Acc: 0.9905 - Val Acc(subset,no ZNE): 0.9823 - Time:

/usr/local/lib/python3.11/dist-packages/pennylane/devices/device_api.py:193: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(
